In [ ]:
#Preparation script for translation 
# weird symbol removal
import os

# Function to clean misencoded symbols from the file content
def clean_misencoded_symbols_in_file(file_path):
    # Open the file and read the contents
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        file_contents = file.read()

    # Replace common misencoded characters (e.g., â€“ for em dash) and similar issues
    cleaned_file_contents = file_contents.replace('â€“', '–')  # Replacing "â€“" with actual em dash
    cleaned_file_contents = cleaned_file_contents.replace('â€™', '’')  # Replacing "â€™" with actual apostrophe
    cleaned_file_contents = cleaned_file_contents.replace('â€œ', '“')  # Replacing "â€œ" with actual opening quote
    cleaned_file_contents = cleaned_file_contents.replace('â€', '”')  # Replacing "â€" with actual closing quote

    # Write the cleaned contents back to the same file (or a new file if you prefer)
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(cleaned_file_contents)

# Function to process all CSV files in a directory and clean the misencoded symbols
def clean_directory(input_directory):
    # Loop through each file in the directory
    for filename in os.listdir(input_directory):
        if filename.endswith('.csv'):
            file_path = os.path.join(input_directory, filename)
            print(f"Cleaning file: {file_path}")
            
            # Clean the misencoded symbols in the file
            clean_misencoded_symbols_in_file(file_path)
            print(f"Finished cleaning file: {file_path}")

# Example usage
if __name__ == "__main__":
    input_directory = 'raw_data'  # Replace with the directory containing your CSV files
    clean_directory(input_directory)


In [ ]:
#Translation script
#Used Utrecth University High processing power computers on remote desktop
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from langdetect import detect
import os

# Initialize MarianMT model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = 'Helsinki-NLP/opus-mt-mul-en'  # Use the correct model for your language pair
model = MarianMTModel.from_pretrained(model_name).to(device)
tokenizer = MarianTokenizer.from_pretrained(model_name)

# Function to detect if the text is in English
def is_english(text):
    try:
        return detect(text) == 'en'
    except Exception as e:
        print(f"Language detection error for text: {text[:50]}... Error: {e}")
        return False

# Function to translate each sentence in a batch
def translate_batch(sentences):
    # Tokenize all sentences in a batch
    tokens = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)

    # Generate translations for the entire batch
    translated_tokens = model.generate(tokens['input_ids'], max_length=512, num_beams=5, no_repeat_ngram_size=2)
    translated_texts = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

    # Calculate token counts for each sentence
    original_tokens_counts = [len(tokens['input_ids'][i]) for i in range(len(sentences))]
    translated_tokens_counts = [len(tokenizer.encode(t, return_tensors='pt')[0]) for t in translated_texts]

    return translated_texts, original_tokens_counts, translated_tokens_counts

# Function to process and translate a column of the DataFrame
def translate_columns(df, columns=['Quote', 'Subject', 'Question', 'Answer'], batch_size=10):
    for column in columns:
        if column in df.columns:
            print(f"\nTranslating column: {column}")
            for index, row in df.iterrows():
                text = row[column]

                if pd.isna(text) or text.strip() == "":
                    continue

                print(f"\nStarting translation for text at index {index}...")

                # Count sentences and tokens before translation
                sentences = [s.strip() for s in text.split('.') if s.strip()]
                original_sentence_count = len(sentences)
                original_token_count = sum(len(tokenizer.encode(sentence.strip())) for sentence in sentences if sentence.strip())

                # Initialize variables for tracking
                translated_text = ""
                total_translated_tokens = 0
                sentences_translated = 0

                print(f"Original sentence count: {original_sentence_count}, Original token count: {original_token_count}")

                # Check if the first sentence is in English and skip entire row if true
                if is_english(sentences[0].strip()):
                    print(f"Skipping translation for row {index} because the first sentence is in English.")
                    continue

                # Process sentences in batches
                batched_sentences = [sentences[i:i + batch_size] for i in range(0, len(sentences), batch_size)]
                
                for batch in batched_sentences:
                    translated_batch, original_tokens, translated_tokens = translate_batch(batch)
                    
                    for t, orig_tokens, trans_tokens in zip(translated_batch, original_tokens, translated_tokens):
                        translated_text += t + ". "
                        total_translated_tokens += trans_tokens
                        sentences_translated += 1
                    print(f"Batch translation completed. Sentences translated: {sentences_translated}/{original_sentence_count}.")
                
                # Update the DataFrame with the translated text
                df.at[index, column] = translated_text.strip()

                # Log the final token counts and sentence counts
                print(f"Translation completed for index {index}. Sentences translated: {sentences_translated}/{original_sentence_count}. Total original tokens: {original_token_count}, Total translated tokens: {total_translated_tokens}")

    return df

# Function to process CSV: Translate and save the results
def process_csv(input_file_path, output_directory, output_filename=None):
    encoding = 'utf-8'  # Assuming UTF-8 encoding for simplicity
    try:
        # Attempt to load the CSV, skipping bad lines (this is the key change for handling malformed data)
        df = pd.read_csv(input_file_path, encoding=encoding, error_bad_lines=False, warn_bad_lines=True)
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return

    # Translate specified columns
    df = translate_columns(df)

    if output_filename:
        output_file_path = os.path.join(output_directory, output_filename)
    else:
        output_file_path = os.path.join(output_directory, f'translated_{os.path.basename(input_file_path)}')

    os.makedirs(output_directory, exist_ok=True)

    try:
        df.to_csv(output_file_path, index=False, encoding='utf-8')
        print(f"Saved translated file: {output_file_path}")
    except Exception as e:
        print(f"Error saving CSV file: {e}")


In [ ]:
# last translations
import pandas as pd
import nltk
import time
import os
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from deep_translator import GoogleTranslator
import langdetect

# Ensure necessary resources are downloaded
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

print("✅ NLTK resources are up-to-date. Starting the script...")

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Define file path
file_path = r"C:\Users\pablo\OneDrive\Escritorio\last_translations\07.xlsx"

# Load the Excel file
print("🔄 Loading Excel file...")
df = pd.read_excel(file_path, sheet_name="Sheet1", dtype=str)  # Load data as strings to preserve formatting
print(f"✅ Excel file loaded successfully. Total rows: {len(df)}. Columns detected: {df.shape[1]}")

# Ensure 4th column (index 3) is always processed
columns_to_translate = [3]  # 4th column
if df.shape[1] > 7:  # Check if 8th column (index 7) exists
    columns_to_translate.append(7)

print(f"🔄 Columns to be translated: {columns_to_translate}")

# List of all 24 official EU languages (for detection)
eu_languages = {
    "bg", "cs", "da", "de", "el", "en", "es", "et", "fi", "fr", "ga", "hr", "hu",
    "it", "lt", "lv", "mt", "nl", "pl", "pt", "ro", "sk", "sl", "sv"
}

# Function to translate long text in chunks
def translate_long_text(text, row_index, col_index):
    if isinstance(text, str) and text.strip():
        try:
            detected_lang = langdetect.detect(text)
            if detected_lang in eu_languages and detected_lang != "en":
                print(f"🌍 Translating row {row_index+1}, column {col_index+1}...")

                # Split text into 4900-character chunks (Google limit is 5000)
                chunk_size = 4900  
                chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

                # Translate each chunk separately (API safety)
                translated_chunks = []
                for chunk in chunks:
                    translated_chunks.append(GoogleTranslator(source="auto", target="en").translate(chunk))
                    time.sleep(1)  # Prevent Google API blocking

                return " ".join(translated_chunks)
        except Exception as e:
            print(f"⚠️ Translation error at row {row_index+1}, column {col_index+1}: {str(e)}")
    return text

# Function to remove stopwords and lemmatize
def preprocess_text(text):
    if isinstance(text, str):
        words = nltk.word_tokenize(text)
        processed_words = [lemmatizer.lemmatize(word) for word in words if word.lower() not in stop_words]
        return ' '.join(processed_words)
    return text

# Function to clean text (remove symbols)
def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # Remove symbols and keep alphanumeric
        return text.lower().strip()  # Convert to lowercase and strip whitespace
    return text

# Function to remove "00:00:00" from column 5 (index 4)
def clean_time(text):
    if isinstance(text, str):
        return text.replace(" 00:00:00", "").strip()  # Remove unwanted time part
    return text

# Apply translations and processing
start_time = time.time()
print("🔄 Starting translations for 4th and 8th columns (if exists)...")

translated_data = df.copy()  # Copy DataFrame to prevent in-place modifications
for col_index in columns_to_translate:
    for row_index in range(len(df)):
        translated_data.iloc[row_index, col_index] = translate_long_text(df.iloc[row_index, col_index], row_index, col_index)

print("✅ Translations complete.")

# Apply preprocessing (stopword removal and lemmatization)
print("🔄 Starting preprocessing...")
for col_index in columns_to_translate:
    translated_data.iloc[:, col_index] = translated_data.iloc[:, col_index].apply(preprocess_text)
print("✅ Preprocessing complete.")

# Apply cleaning on 4th column (remove symbols)
print("🔄 Cleaning 4th column (removing symbols)...")
translated_data.iloc[:, 3] = translated_data.iloc[:, 3].apply(clean_text)
print("✅ 4th column cleaned.")

# Apply cleaning on 5th column (remove "00:00:00")
if df.shape[1] > 4:
    print("🔄 Cleaning 5th column (removing '00:00:00')...")
    translated_data.iloc[:, 4] = translated_data.iloc[:, 4].apply(clean_time)
    print("✅ 5th column cleaned.")

# Save the processed dataframe as a CSV file
print("🔄 Saving processed file...")
output_path = os.path.splitext(file_path)[0] + ".csv"
translated_data.to_csv(output_path, index=False, encoding="utf-8")
print(f"\n✅ Processing complete. File saved as {output_path}")

# Log completion
end_time = time.time()
elapsed_time = end_time - start_time
print(f"⏱️ Total time taken: {elapsed_time:.2f} seconds")
print(f"📊 Total rows processed: {len(df)}")
